In [ ]:
# ============================================
# Radar-Based Pose Estimation (Project Version)
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# root = sessionID+'/alignment/'
root  = './dataset/'
index = root + '00200'

import matplotlib.pyplot as plt
import matplotlib.patches as patches
# joints connections for 2d keypoints
connections = np.array([[13, 15], [11, 13], [14, 16], [12, 14], [11, 12],
                [5, 11], [6, 12], [5, 6], [5, 7], [6, 8], [7, 9],
                [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
                [3, 5], [4, 6]])

# Meta info
meta_data  = np.load(index + '_meta.npz')
global_id  = meta_data['global_frame_id']
# Horizontal/Vertical heatmaps
radar_data = np.load(index + '_radar.npz')
hori_raw   = radar_data['hm_hori']
vert_raw   = radar_data['hm_vert']
# 2D Bounding Boxes
bbox_data  = np.load(index + '_bbox.npz')
bbox_i     = bbox_data['bbox_i']
bbox_hori  = bbox_data['bbox_hori']
bbox_vert  = bbox_data['bbox_vert']
# 2D keypoints
pose_data  = np.load(index + '_pose.npz')
kp         = pose_data['kp']
# Segmentation masks
mask_data  = np.load(index + '_mask.npz')
mask       = mask_data['mask']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------
# 1. Prepare Radar Input
# --------------------------------------------
# Convert raw radar heatmaps to float32 arrays
hori = hori_raw.astype(np.float32)
vert = vert_raw.astype(np.float32)

# Replace NaN/Inf values with zeros for stability
hori = np.nan_to_num(hori, nan=0.0, posinf=0.0, neginf=0.0)
vert = np.nan_to_num(vert, nan=0.0, posinf=0.0, neginf=0.0)

# Clip extreme values at the 99th percentile
hori = np.clip(hori, 0, np.percentile(hori, 99))
vert = np.clip(vert, 0, np.percentile(vert, 99))

# Normalize each map to [0, 1] range
hori = hori / (np.max(hori) + 1e-6)
vert = vert / (np.max(vert) + 1e-6)

# Check whether any NaN values remain
print("hori NaN:", np.isnan(hori).any())
print("vert NaN:", np.isnan(vert).any())

# Combine horizontal + vertical heatmaps as 2-channel input
radar_input  = np.stack([hori, vert], axis=0)
# Normalize
radar_tensor = torch.tensor(radar_input, dtype=torch.float32).unsqueeze(0).to(device)

if np.isnan(radar_input).any():
    raise ValueError("NaN still present in radar_input!")

# --------------------------------------------
# 2. Soft-Argmax Layer
# --------------------------------------------
class SoftArgmax2D(nn.Module):
    def forward(self, heatmaps):
        B, C, H, W = heatmaps.shape
        # Improve numerical stability by shifting each heatmap so its maximum value is 0
        heatmaps = heatmaps - heatmaps.amax(dim=(2,3), keepdim=True)
        # Flatten spatial dimensions (H, W) and convert each heatmap into a probability distribution
        heatmaps = F.softmax(heatmaps.view(B, C, -1), dim=-1)
        indices  = torch.arange(H*W, device=heatmaps.device).float()
        x = indices % W
        y = torch.div(indices, W, rounding_mode='floor')
        return torch.stack([
            torch.sum(heatmaps * x, dim=-1),
            torch.sum(heatmaps * y, dim=-1),
        ], dim=-1)

# --------------------------------------------
# 3. Custom CNN
# --------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)

class PoseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(2,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU())
        self.res  = ResidualBlock(64)
        self.dec1 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU())
        self.dec2 = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU())
        self.out  = nn.Conv2d(32, 17, 1)
        self.soft_argmax = SoftArgmax2D()

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.res(x)
        x = self.dec1(x)
        x = self.dec2(x)
        heatmaps = self.out(x)
        return heatmaps, self.soft_argmax(heatmaps)

model_cnn = PoseCNN().to(device)

# --------------------------------------------
# 4. Baseline Model (ResNet-18)
# --------------------------------------------
from torchvision.models import resnet18

class ResNetPose(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone         = resnet18(weights=None)
        self.backbone.conv1   = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc      = nn.Linear(512, 17*2)

    def forward(self, x):
        return self.backbone(x).view(-1, 17, 2)

model_resnet = ResNetPose().to(device)

# --------------------------------------------
# 5. Loss + GT Heatmaps
# --------------------------------------------
criterion = nn.BCEWithLogitsLoss()

def create_heatmaps(gt_kp, H=256, W=128, sigma=4):
    xx, yy   = np.meshgrid(np.arange(W), np.arange(H))
    heatmaps = np.zeros((17, H, W), dtype=np.float32)
    for i in range(17):
        x, y, v = gt_kp[i]
        if v > 0.3:
            x, y = int(np.clip(x, 0, W-1)), int(np.clip(y, 0, H-1))
            heatmaps[i] = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
    return torch.tensor(heatmaps, dtype=torch.float32)

gt_kp       = kp[0]
gt_heatmaps = create_heatmaps(gt_kp).unsqueeze(0).to(device)

# --------------------------------------------
# 6. Forward Pass
# --------------------------------------------
heatmaps_pred, coords_pred = model_cnn(radar_tensor)

print("Heatmap NaNs:", torch.isnan(heatmaps_pred).any())
print("Coords NaNs:",  torch.isnan(coords_pred).any())

coords_resnet = model_resnet(radar_tensor)
loss = criterion(heatmaps_pred, gt_heatmaps)
print("Loss:", loss.item())

coords_pred   = coords_pred[0].detach().cpu().numpy()
coords_resnet = coords_resnet[0].detach().cpu().numpy()

print("Pred NaN:",   np.isnan(coords_pred).any())
print("ResNet NaN:", np.isnan(coords_resnet).any())

# --------------------------------------------
# 7. Metrics
# --------------------------------------------
def mae(gt, pred):
    return np.mean(np.linalg.norm(gt[:,:2] - pred, axis=1))

def pck(gt, pred, thresh=0.05, img_size=256):
    return np.mean(np.linalg.norm(gt[:,:2] - pred, axis=1) < thresh * img_size)

def precision_recall_f1(gt, pred, thresh=5):
    dist      = np.linalg.norm(gt[:,:2] - pred, axis=1)
    tp        = np.sum(dist < thresh)
    fp        = np.sum(dist >= thresh)
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (len(gt) + 1e-6)
    f1        = 2 * (precision * recall) / (precision + recall + 1e-6)
    return precision, recall, f1

def oks(gt, pred):
    return np.mean([np.exp(-np.linalg.norm(gt[i,:2] - pred[i])**2 / 100) for i in range(17)])

for name, pred in [("Custom CNN", coords_pred), ("ResNet-18", coords_resnet)]:
    print(f"\n=== {name} ===")
    print("MAE:", mae(gt_kp, pred))
    print("PCK:", pck(gt_kp, pred))
    print("OKS:", oks(gt_kp, pred))
    p, r, f1 = precision_recall_f1(gt_kp, pred)
    print("Precision:", p, "Recall:", r, "F1:", f1)

# --------------------------------------------
# 8. Robustness Test (Noise)
# --------------------------------------------
noisy_input = radar_tensor + torch.randn_like(radar_tensor) * 0.1
_, coords_noisy = model_cnn(noisy_input)
coords_noisy = coords_noisy[0].detach().cpu().numpy()
print("\n=== Robustness Test (Noise) ===")
print("PCK:", pck(gt_kp, coords_noisy))